In [ ]:
import torch
print("GPU:", torch.cuda.get_device_name(0))
print("VRAM:", round(torch.cuda.get_device_properties(0).total_memory/1024**3, 1), "GB")

In [ ]:
!pip install -q -U transformers peft datasets evaluate accelerate sentencepiece

In [ ]:
import transformers
print(transformers.__version__)

In [ ]:
import sys
sys.path.insert(0, '/kaggle/input/phobert-scripts')
print("Chay xong")

In [ ]:
import os
for root, dirs, files in os.walk('/kaggle/input'):
    for f in files:
        print(os.path.join(root, f))

In [ ]:
!python /kaggle/input/datasets/tntminh1299/script/train_phobert_5cls.py \
  --train_csv       /kaggle/input/datasets/tntminh1299/mydata/train.csv \
  --test_csv        /kaggle/input/datasets/tntminh1299/mydata/test.csv \
  --output_dir      /kaggle/working/phobert_output2 \
  --batch_size      64 \
  --max_len         160 \
  --epochs          15 \
  --lr              4e-5 \
  --weight_decay    0.05 \
  --hidden_dropout  0.2 \
  --attn_dropout    0.2 \
  --label_smoothing 0.1 \
  --eval_strategy   epoch \
  --class_weight \
  --normalize

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('/kaggle/working/phobert_output2/training_log.csv')

train_log = df[df.phase == 'train'].dropna(subset=['loss_ema'])
val_log   = df[df.phase == 'val'].dropna(subset=['loss'])

plt.figure(figsize=(10, 5))
plt.plot(train_log.epoch, train_log.loss_ema, label='Train Loss (EMA)', color='steelblue', linewidth=1.5)
plt.plot(val_log.epoch,   val_log.loss,       label='Val Loss',         color='tomato', marker='o', linewidth=2)
plt.xlabel('Epoch')
plt.ylabel('Loss')
plt.title('PhoBERT Fine-tune — Train vs Val Loss')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('/kaggle/working/phobert_output2/train_loss.png', dpi=150)
plt.show()

In [ ]:
import pandas as pd

df = pd.read_csv('/kaggle/working/phobert_output2/training_log.csv')
print(df.columns.tolist())
print(df.head(20).to_string())

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

df = pd.read_csv('/kaggle/working/phobert_output2/training_log.csv')

# Train: dùng loss thực, bỏ EMA
train_log = df[df.phase == 'train'].dropna(subset=['loss']).copy()

# Val: bỏ duplicate
val_log = df[df.phase == 'val'].dropna(subset=['loss']).drop_duplicates(subset=['epoch']).copy()

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

# --- Loss ---
axes[0].plot(train_log.epoch, train_log.loss, label='Train Loss', color='steelblue', linewidth=1.5)
axes[0].plot(val_log.epoch,   val_log.loss,   label='Val Loss',   color='tomato', marker='o', linewidth=2)
axes[0].set_title('Loss'); axes[0].set_xlabel('Epoch'); axes[0].legend(); axes[0].grid(True, alpha=0.3)

# --- Val Macro F1 ---
axes[1].plot(val_log.epoch, val_log.macro_f1, color='seagreen', marker='o', linewidth=2)
axes[1].set_title('Val Macro F1'); axes[1].set_xlabel('Epoch'); axes[1].grid(True, alpha=0.3)

plt.suptitle('PhoBERT Fine-tune (10 epochs)', fontsize=13)
plt.tight_layout()
plt.savefig('/kaggle/working/phobert_output2/train_curve.png', dpi=150)
plt.show()

In [ ]:
import os

output_dir = '/kaggle/working/phobert_output2'

# Liệt kê tất cả checkpoints còn lại
for item in sorted(os.listdir(output_dir)):
    print(item)

In [ ]:
import pandas as pd

df = pd.read_csv('/kaggle/working/phobert_output2/training_log.csv')
val = df[df.phase == 'val'].dropna(subset=['macro_f1']).drop_duplicates(subset=['epoch'])
print(val[['epoch','loss','accuracy','macro_f1']].to_string(index=False))

In [ ]:
import shutil
shutil.make_archive('/kaggle/working/phobert_best_model', 'zip', '/kaggle/working/phobert_output2')
print("Done! Vào tab Output bên phải để download.")

In [ ]:
from IPython.display import display, FileLink

# Kiểm tra file tồn tại và size
import os
path = '/kaggle/working/phobert_best_model.zip'
if os.path.exists(path):
    size = os.path.getsize(path) / 1024**2
    print(f"File tồn tại: {size:.1f} MB")
    display(FileLink(path))
else:
    print("File chưa tồn tại, chạy lại cell tạo zip trước")